In [ ]:
import rasterio
import numpy as np 
import warnings 
from PIL import Image 
import matplotlib.pyplot as plt 
warnings.simplefilter("ignore")

## 1. filter Tiff files 

In [ ]:
def filter_tiff(tiff_array: np.array, threshold: float, null_value=0):
    arr = np.where(tiff_array <= threshold, tiff_array, null_value)
    arr[arr>0] = 1 # treat it as binary 
    return arr 

In [ ]:
#load 
img = rasterio.open("037_PP-TPE_23_h_0001_0003pag_db0100_vol_000250.tiff")
arr = img.read()

In [ ]:
#filter 
percent = 0.6
filtered_arr = filter_tiff(arr,percent*np.max(arr),0)

np.savez_compressed("filtered_037_PP-TPE_23_h_0001_0003pag_db0100_vol_000250.npz",filtered_arr)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 6), dpi=100)
axes[0].imshow(Image.fromarray(-arr[0]), cmap="cividis")
axes[1].imshow(Image.fromarray(-filtered_arr[0]), cmap="cividis")
[ax.set_yticks([]) for ax in axes]
[ax.set_xticks([]) for ax in axes]

axes[0].set_title("unfiltered")
axes[1].set_title("filtered")

##

## 2. multiprocessing

In [ ]:
import glob
import tqdm_pathos
import natsort


In [ ]:
files = sorted(glob.glob("*.tiff"), key=natsort.natsort_key)


def multiprocessing_function(file, threshold):
    img = rasterio.open(file)
    arr = img.read()
    arr = filter_tiff(arr, threshold * np.max(arr), 0)
    outfilename = file.replace(".tiff", ".npz")
    np.savez_compressed(outfilename, arr)


threshold = 0.6
tqdm_pathos.map(multiprocessing_function, files, threshold, n_cpus=12)
